# Unidad 4 · Cuaderno 05 · Aplicaciones en investigación

**Modelación y Simulación Computacional** · Maestría en Ingeniería, Universidad de Sucre, periodo 2026-2

**Unidad 4.** Validación, interpretación y comunicación de resultados
· **Subtema del plan 4.5**

Este cuaderno ejecuta lo que el libro expone en la subsección 4.6.2. El texto no
repite la teoría, remite a ella por número de definición, de teorema, de
ejemplo, de listado, de tabla o de figura, y se ocupa de reproducir los
resultados publicados y de verificarlos.

**Autor.** Prof. Daniel Otero Meza, Ing., Ph.D.

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad4/U4_05_aplicaciones_en_investigacion.ipynb)

## Objetivos de aprendizaje

1. Formular una pregunta de investigación que el modelo convierta en una predicción cuantitativa refutable, como pide la sección 4.6.2.
2. Recorrer el ciclo completo de verificación, calibración y validación sobre un caso propio de la Facultad.
3. Declarar el criterio de aceptación antes de mirar los datos reservados, según el Algoritmo 4.3 del libro.
4. Propagar la incertidumbre de los parámetros a la predicción que sostiene la decisión, con la ley de primer orden y con Monte Carlo.
5. Construir la figura única que sostiene la conclusión y redactar la afirmación defendible con su dominio de validez.

## Puesta a punto

In [ ]:
# Puesta a punto. Detecta el entorno e instala solo lo que falte.
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict[str, str]) -> None:
    """Instala los paquetes cuyo módulo no se encuentre en el entorno."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("Entorno listo. Colab:", EN_COLAB)

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

PALETA = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
           "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 10, "axes.grid": True, "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=list(PALETA.values())),
})

# Bandera de los ejercicios guiados. En la versión de trabajo vale False
# para que el cuaderno corra completo aunque falten celdas por resolver.
REVISAR = False


def verificar(nombre: str, obtenido, esperado: float,
              tol: float = 1.0e-3) -> bool:
    """Compara un resultado con el valor esperado sin detener el cuaderno."""
    if obtenido is None or (isinstance(obtenido, float) and np.isnan(obtenido)):
        print(f"[pendiente] {nombre}, la celda marcada COMPLETE sigue sin resolver")
        return False
    escala = abs(esperado) if esperado != 0.0 else 1.0
    error = abs(float(obtenido) - esperado) / escala
    estado = "ok" if error <= tol else "revisar"
    print(f"[{estado}] {nombre}, obtenido {float(obtenido):.6g}, "
          f"esperado {esperado:.6g}, error relativo {error:.2e}")
    if REVISAR:
        assert error <= tol, f"{nombre} no coincide con el valor esperado"
    return error <= tol


print("Semilla del curso:", SEMILLA)

In [ ]:
# Acceso a datos/ que funciona en Colab y en local, sin rutas absolutas.
# Si la carpeta no viaja con el cuaderno, las series se reconstruyen con la
# semilla del curso y con las cifras que el libro publica.

def carpeta_datos() -> Path:
    """Ubica datos/ subiendo por el árbol, o la crea junto al cuaderno."""
    base = Path.cwd()
    for nivel in [base, *base.parents][:4]:
        for candidata in (nivel / "datos", nivel / "03_cuadernos" / "datos"):
            if candidata.is_dir():
                return candidata
    destino = base / "datos"
    destino.mkdir(parents=True, exist_ok=True)
    return destino


def _cinetica_monod() -> pd.DataFrame:
    return pd.DataFrame({
        "S_g_L": [0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 12.0, 18.0, 25.0, 35.0],
        "mu_1_h": [0.0702, 0.1248, 0.1919, 0.2496, 0.2761,
                   0.3205, 0.3689, 0.3709, 0.3770, 0.3773]})


def _secado_calibracion() -> pd.DataFrame:
    t = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0,
                  4.0, 5.0, 6.0, 7.0, 8.0])
    gen = np.random.default_rng(SEMILLA)
    mr = np.round(np.exp(-0.350 * t**1.15) + gen.normal(0.0, 0.008, t.size), 4)
    return pd.DataFrame({"t_h": t, "MR": mr})


def _secado_validacion() -> pd.DataFrame:
    t = np.array([0.5, 1.0, 1.5, 2.0, 2.75, 3.5, 4.5, 5.5, 6.5, 8.0])
    gen = np.random.default_rng(SEMILLA + 1)
    mr = np.round(np.exp(-0.362 * t**1.15) + gen.normal(0.0, 0.008, t.size), 4)
    return pd.DataFrame({"t_h": t, "MR": mr})


def _caudal_mensual() -> pd.DataFrame:
    obs = [6.21, 5.01, 4.30, 9.39, 14.85, 18.28, 16.85, 17.12, 21.51, 22.75,
           19.33, 12.13, 7.65, 6.14, 4.54, 8.58, 14.73, 21.82, 14.36, 17.57,
           24.54, 31.26, 21.30, 11.27, 9.71, 6.09, 5.39, 10.86, 23.94, 20.68,
           19.39, 22.16, 32.76, 38.82, 21.33, 12.99, 5.97, 5.83, 5.07, 8.20,
           18.23, 18.93, 12.72, 16.63, 22.38, 27.76, 18.00, 12.27]
    sim = [7.70, 4.70, 4.43, 7.83, 14.55, 16.00, 15.33, 17.97, 20.37, 24.45,
           17.19, 13.27, 10.40, 6.93, 5.40, 10.19, 16.72, 22.35, 13.78, 17.44,
           24.20, 29.67, 17.56, 13.20, 12.94, 2.59, 5.60, 12.74, 21.94, 16.05,
           18.74, 17.99, 27.76, 24.48, 15.41, 12.64, 6.83, 6.11, 5.24, 11.99,
           19.88, 20.51, 12.28, 17.89, 19.38, 19.28, 14.48, 8.00]
    return pd.DataFrame({"mes": np.arange(1, 49),
                         "periodo": ["calibracion"] * 24 + ["validacion"] * 24,
                         "Q_obs_m3_s": obs, "Q_sim_m3_s": sim})


def _arreglo_fotovoltaico() -> pd.DataFrame:
    return pd.DataFrame({"configuracion": ["Base", "Optima", "Sombreado"],
                         "H_kWh_m2": [1980.0, 2035.0, 1910.0],
                         "u_rel_H": [0.04, 0.04, 0.04]})


def _entradas_vertedero() -> pd.DataFrame:
    return pd.DataFrame({"magnitud": ["C_d", "b", "h"],
                         "unidad": ["1", "m", "m"],
                         "valor": [0.620, 0.500, 0.150],
                         "u_tipica": [0.015, 0.0010, 0.0015]})


CONSTRUCTORES = {
    "cinetica_monod.csv": _cinetica_monod,
    "secado_maiz_calibracion.csv": _secado_calibracion,
    "secado_maiz_validacion.csv": _secado_validacion,
    "caudal_mensual.csv": _caudal_mensual,
    "arreglo_fotovoltaico.csv": _arreglo_fotovoltaico,
    "entradas_vertedero.csv": _entradas_vertedero,
}

CARPETA_DATOS = carpeta_datos()


def leer_datos(nombre: str) -> pd.DataFrame:
    """Lee un archivo de datos/ y lo reconstruye si no está presente."""
    ruta = CARPETA_DATOS / nombre
    if not ruta.exists():
        CONSTRUCTORES[nombre]().to_csv(ruta, index=False)
    return pd.read_csv(ruta)


print("Carpeta de datos:", CARPETA_DATOS.name)

## 1. La pregunta y la decisión que apoya

La sección 4.6.2 del libro recuerda que en un trabajo de
investigación el modelo rara vez es el resultado, casi siempre es el
instrumento. Su función más frecuente consiste en convertir una
hipótesis cualitativa en una predicción cuantitativa que el
experimento pueda refutar, lo cual obliga a formularla antes de
medir. Su segunda función consiste en decidir dónde medir.

El caso de este cuaderno pertenece a la agroindustria, uno de los
cinco dominios que el libro enumera. Un grupo de poscosecha necesita
fijar el tiempo de residencia de un secador de maíz de capa delgada
que opera a 60 grados Celsius, y quiere saber con qué margen puede
comprometerlo.

La pregunta de investigación queda así. ¿Cuánto tiempo se necesita
para llevar la razón de humedad del maíz hasta 0.10 a 60 grados
Celsius, con qué incertidumbre, y sigue valiendo esa cifra para un
lote que el modelo nunca vio?

La decisión que apoya es concreta, el ajuste del tiempo de residencia
del equipo, que se traduce en velocidad de la banda y en consumo
energético.

In [ ]:
PREGUNTA = ("¿Cuánto dura el secado del maíz hasta una razón de "
            "humedad de 0.10 a 60 grados Celsius, con qué margen, y "
            "vale la cifra para un lote independiente?")
DECISION = ("Fijar el tiempo de residencia del secador de capa "
            "delgada y con él la velocidad de la banda.")
MR_OBJETIVO = 0.10

# Criterio de aceptación declarado ANTES de mirar el lote reservado,
# según el paso primero del Algoritmo 4.3 del libro.
CRITERIO = {"RMSE_maximo": 0.025, "PBIAS_maximo": 10.0,
            "NSE_minimo": 0.95}

print("Pregunta:", PREGUNTA)
print("Decisión que apoya:", DECISION)
print("Criterio de aceptación declarado de antemano:", CRITERIO)

## 2. Los datos, dos campañas independientes

El archivo `datos/secado_maiz_calibracion.csv` recoge la campaña del
lote A, que es la del Ejemplo 4.4 del libro, con trece instantes
entre 0.25 h y 8 h. El archivo `datos/secado_maiz_validacion.csv`
recoge el lote B, una campaña independiente en las mismas condiciones
de aire y con otra malla de tiempos.

La Definición 4.7 del libro exige que el conjunto de validación no
intervenga en ninguna etapa de la estimación, ni siquiera en la
selección de la estructura. Aquí se cumple por construcción, porque
la estructura se escogió en el cuaderno U4_01 usando solo el lote A.

In [ ]:
calibracion = leer_datos("secado_maiz_calibracion.csv")
validacion = leer_datos("secado_maiz_validacion.csv")

print("Lote A, campaña de calibración")
print(calibracion.to_string(index=False))
print("\nLote B, campaña independiente de validación")
print(validacion.to_string(index=False))
print(f"\nNinguno de los {len(validacion)} instantes del lote B "
      f"coincide con los del lote A: "
      f"{set(validacion['t_h']).isdisjoint(set(calibracion['t_h']))}")

## 3. Verificación del código antes de estimar nada

El orden de la Definición 4.1 del libro es estricto. Antes de ajustar
un solo parámetro conviene comprobar que el integrador resuelve la
ecuación que se escribió. El modelo de Page tiene forma cerrada, de
modo que sirve de caso de referencia para verificar un integrador que
después se usará con cinéticas sin solución analítica.

La ecuación diferencial equivalente al modelo de Page se obtiene
derivando su forma cerrada, y el integrador de Runge y Kutta de
cuarto orden debe reproducirla con orden observado cercano a cuatro.

In [ ]:
def page(t, k, n):
    """Modelo de Page, razón de humedad adimensional con t en horas."""
    return np.exp(-k * t**n)


def derivada_page(t, mr, k, n):
    """Ecuación diferencial equivalente al modelo de Page, en 1/h."""
    return -k * n * np.maximum(t, 1.0e-12)**(n - 1.0) * mr


def integrar_rk4(k, n, t_final, pasos):
    """Runge y Kutta de cuarto orden con paso constante."""
    h = t_final / pasos
    t, mr = 0.0, 1.0
    for _ in range(pasos):
        k1 = derivada_page(t, mr, k, n)
        k2 = derivada_page(t + h / 2, mr + h * k1 / 2, k, n)
        k3 = derivada_page(t + h / 2, mr + h * k2 / 2, k, n)
        k4 = derivada_page(t + h, mr + h * k3, k, n)
        mr += h * (k1 + 2 * k2 + 2 * k3 + k4) / 6
        t += h
    return mr


def orden_observado(h, e):
    """Orden por pares y por regresión logarítmica, Listado 4.2."""
    h, e = np.asarray(h, float), np.asarray(e, float)
    p_par = np.log(e[:-1] / e[1:]) / np.log(h[:-1] / h[1:])
    p_reg = float(np.polyfit(np.log(h), np.log(e), 1)[0])
    return p_par, p_reg


K_PRUEBA, N_PRUEBA, T_PRUEBA = 0.35, 2.0, 4.0
referencia = page(T_PRUEBA, K_PRUEBA, N_PRUEBA)
pasos_malla = [40, 80, 160, 320, 640]
pasos_h, errores = [], []
for pasos in pasos_malla:
    pasos_h.append(T_PRUEBA / pasos)
    errores.append(abs(integrar_rk4(K_PRUEBA, N_PRUEBA, T_PRUEBA, pasos)
                       - referencia))

p_par, p_reg = orden_observado(pasos_h, errores)
print("pasos   h (h)     error        orden")
for i, pasos in enumerate(pasos_malla):
    orden = f"{p_par[i - 1]:.3f}" if i else "---"
    print(f"{pasos:5d}  {pasos_h[i]:.5f}  {errores[i]:.3e}  {orden:>7}")
print(f"\nPendiente global {p_reg:.3f}, frente al orden formal 4 del "
      f"esquema.")
assert abs(p_reg - 4.0) < 0.1
print("El integrador queda verificado antes de estimar ningún parámetro.")

El exponente de la prueba se tomó entero y mayor que uno para que la
derivada sea suave en el origen. Con un exponente fraccionario el
término de potencia tiene derivada infinita en el instante inicial y
el orden observado cae, defecto real del problema y no del
integrador, que conviene conocer antes de usarlo con los parámetros
ajustados.

## 4. Calibración con el lote A y su precisión

La Definición 4.4 del libro trata la calibración como un problema de
optimización, y el Teorema 4.2 permite acompañar cada parámetro de su
error estándar, su intervalo y su correlación con los demás.

In [ ]:
from scipy.optimize import curve_fit
from scipy.stats import t as t_student

t_cal = calibracion["t_h"].to_numpy()
mr_cal = calibracion["MR"].to_numpy()

theta, cov = curve_fit(page, t_cal, mr_cal, p0=[0.3, 1.0])
n_datos, m_par = t_cal.size, theta.size
ee = np.sqrt(np.diag(cov))
t_critico = t_student.ppf(0.975, n_datos - m_par)
correlacion = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
residuales = mr_cal - page(t_cal, *theta)
suma_cuadrados = float(residuales @ residuales)
s_residual = np.sqrt(suma_cuadrados / (n_datos - m_par))

print(f"k = {theta[0]:.4f} +- {ee[0]:.4f}   (el libro publica 0.3430)")
print(f"n = {theta[1]:.4f} +- {ee[1]:.4f}   (el libro publica 1.1627)")
print(f"Suma de cuadrados {suma_cuadrados:.4e}   (libro 5.815e-04)")
print(f"Desviación residual {s_residual:.4f}   (libro 0.0073)")
print(f"Correlación entre k y n {correlacion:.3f}")
print(f"Intervalo de k: {theta[0] - t_critico * ee[0]:.4f} a "
      f"{theta[0] + t_critico * ee[0]:.4f}")
print(f"Intervalo de n: {theta[1] - t_critico * ee[1]:.4f} a "
      f"{theta[1] + t_critico * ee[1]:.4f}")
assert abs(theta[0] - 0.3430) < 5.0e-5
assert abs(theta[1] - 1.1627) < 1.0e-4
assert abs(suma_cuadrados - 5.815e-4) < 5.0e-7
assert abs(s_residual - 0.0073) < 5.0e-5

## 5. Validación con el lote B, sin retocar un solo parámetro

El paso tercero del Algoritmo 4.3 del libro es taxativo, se ejecuta
el modelo con los parámetros calibrados y no se retoca ninguno. El
criterio de aceptación quedó escrito en la sección 1 de este
cuaderno, antes de mirar el lote reservado.

In [ ]:
def metricas(obs, sim) -> dict[str, float]:
    """Métricas de uso corriente en validación, Listado 4.7."""
    obs, sim = np.asarray(obs, float), np.asarray(sim, float)
    e = sim - obs
    rmse = np.sqrt(np.mean(e**2))
    dif = np.abs(sim - obs.mean()) + np.abs(obs - obs.mean())
    return {"ME": e.mean(), "MAE": np.abs(e).mean(), "RMSE": rmse,
            "PBIAS": 100 * e.sum() / obs.sum(),
            "NSE": 1 - np.sum(e**2) / np.sum((obs - obs.mean())**2),
            "RSR": rmse / obs.std(),
            "d": 1 - np.sum(e**2) / np.sum(dif**2)}


t_val = validacion["t_h"].to_numpy()
mr_val = validacion["MR"].to_numpy()
mr_predicho = page(t_val, *theta)

m_cal = metricas(mr_cal, page(t_cal, *theta))
m_val = metricas(mr_val, mr_predicho)
comparacion = pd.DataFrame({"lote A, calibración": m_cal,
                            "lote B, validación": m_val})
print(comparacion.to_string(float_format=lambda v: f"{v:9.4f}"))

cumple = (m_val["RMSE"] <= CRITERIO["RMSE_maximo"]
          and abs(m_val["PBIAS"]) <= CRITERIO["PBIAS_maximo"]
          and m_val["NSE"] >= CRITERIO["NSE_minimo"])
print(f"\nCriterio declarado: {CRITERIO}")
print(f"Veredicto de aceptación: {cumple}")
assert cumple

### 5.1 Lo que las métricas esconden

El Listado 4.8 del libro examina paridad, tendencia, autocorrelación
y heterocedasticidad. El modelo pasa el criterio, y aun así los
residuales del lote B tienen una firma que las métricas resumen y por
tanto ocultan.

In [ ]:
def diagnostico_residuales(obs, sim) -> dict[str, float]:
    """Paridad, tendencia y estructura del residual, Listado 4.8."""
    obs, sim = np.asarray(obs, float), np.asarray(sim, float)
    r = sim - obs
    return dict(paridad=np.polyfit(obs, sim, 1)[0],
                tendencia=np.polyfit(obs, r, 1)[0],
                autocorrelacion=np.corrcoef(r[:-1], r[1:])[0, 1],
                heterocedasticidad=np.corrcoef(np.abs(r), obs)[0, 1])


diag = diagnostico_residuales(mr_val, mr_predicho)
for nombre, valor in diag.items():
    print(f"{nombre:20s} {valor: .3f}")

residual_val = mr_predicho - mr_val
print(f"\nResidual medio {residual_val.mean(): .4f}, positivo en "
      f"{int(np.sum(residual_val > 0))} de {residual_val.size} instantes.")
print("El modelo sobrestima la razón de humedad del lote B de manera")
print("sistemática, porque ese lote seca algo más rápido. La firma es")
print("de sesgo entre lotes, no de estructura ausente en el modelo.")
assert residual_val.mean() > 0

## 6. La predicción que sostiene la decisión

La pregunta pide el tiempo hasta una razón de humedad de 0.10.
Invertir el modelo de Page da una forma cerrada, de modo que la
incertidumbre de los parámetros se propaga a esa predicción con la
ley de primer orden del Teorema 4.4 y se contrasta con Monte Carlo,
según manda el Algoritmo 4.4.

In [ ]:
def tiempo_objetivo(k, n, mr=MR_OBJETIVO):
    """Inversa del modelo de Page, tiempo en horas."""
    return (-np.log(mr) / k) ** (1.0 / n)


def gradiente_tiempo(k, n, mr=MR_OBJETIVO):
    """Derivadas parciales del tiempo respecto de k y de n, en h."""
    logaritmo = -np.log(mr)
    t = (logaritmo / k) ** (1.0 / n)
    return np.array([t * (-1.0 / (n * k)),
                     t * (-np.log(logaritmo / k) / n**2)])


t_predicho = float(tiempo_objetivo(*theta))
gradiente = gradiente_tiempo(*theta)
u_lineal = float(np.sqrt(gradiente @ cov @ gradiente))

generador = np.random.default_rng(SEMILLA)
muestras = generador.multivariate_normal(theta, cov, 100_000)
tiempos = tiempo_objetivo(muestras[:, 0], muestras[:, 1])
u_monte_carlo = float(tiempos.std(ddof=1))
cobertura = np.percentile(tiempos, [2.5, 97.5])
discrepancia = 100 * (u_monte_carlo - u_lineal) / u_lineal

print(f"Tiempo hasta MR = {MR_OBJETIVO:.2f}: {t_predicho:.3f} h")
print(f"Incertidumbre por primer orden {u_lineal:.4f} h")
print(f"Incertidumbre por Monte Carlo  {u_monte_carlo:.4f} h")
print(f"Discrepancia entre métodos {discrepancia:.2f} por ciento")
print(f"Intervalo de cobertura al 95 por ciento: "
      f"{cobertura[0]:.3f} a {cobertura[1]:.3f} h")
assert abs(discrepancia) < 5.0
print("\nLa linealización es adecuada según el Algoritmo 4.4, de modo")
print("que el intervalo simétrico representa bien la incertidumbre.")

### 6.1 El reporte con cifras significativas coherentes

La sección 4.4.2 del libro exige que la incertidumbre gobierne las
cifras del valor. La celda siguiente aplica la regla y comprueba que
el margen entre lotes, medido en la validación, no quede fuera del
reporte.

In [ ]:
def formato_valor_incertidumbre(valor, u, cifras=2, unidad=""):
    """Texto con el valor y la incertidumbre a la misma posición decimal."""
    if u == 0.0 or not np.isfinite(u):
        return f"{valor} {unidad}".strip()
    posicion = int(cifras - 1 - np.floor(np.log10(abs(u))))
    decimales = max(posicion, 0)
    sufijo = f" {unidad}" if unidad else ""
    return (f"{round(valor, posicion):.{decimales}f} +- "
            f"{round(u, posicion):.{decimales}f}{sufijo}")


# Al margen del ajuste se suma el sesgo entre lotes que la validación
# midió, porque la decisión se toma sobre lotes que el modelo no vio.
t_lote_b = float(tiempo_objetivo(
    *curve_fit(page, validacion["t_h"], validacion["MR"],
               p0=theta)[0]))
sesgo_entre_lotes = abs(t_lote_b - t_predicho)
u_total = float(np.sqrt(u_lineal**2 + sesgo_entre_lotes**2))

print("Solo con la incertidumbre del ajuste:")
print("  ", formato_valor_incertidumbre(t_predicho, u_lineal, 2, "h"))
print(f"Tiempo que el propio lote B habría exigido: {t_lote_b:.3f} h")
print(f"Sesgo entre lotes {sesgo_entre_lotes:.3f} h")
print("Con el sesgo entre lotes incorporado:")
print("  ", formato_valor_incertidumbre(t_predicho, u_total, 2, "h"))
print("\nReportar solo la primera cifra sería honesto con el ajuste y")
print("engañoso con la decisión, porque el equipo secará otros lotes.")
assert u_total > u_lineal

## 7. La figura única que sostiene la conclusión

La sección 4.6.1 del libro recomienda organizar la exposición
alrededor de una sola figura. Esa figura debe mostrar la evidencia
que sostiene la afirmación, no todo lo que se hizo. Aquí muestra las
dos campañas, el modelo calibrado con su banda de confianza y la
predicción que la decisión necesita, con su margen.

In [ ]:
fig, (izq, der) = plt.subplots(
    1, 2, figsize=(10.6, 4.2),
    gridspec_kw={"width_ratios": [1.55, 1.0]})

malla_t = np.linspace(0.05, 8.6, 400)
ajuste = page(malla_t, *theta)
jacobiano = np.vstack([
    -malla_t**theta[1] * ajuste,
    -theta[0] * malla_t**theta[1] * np.log(malla_t) * ajuste]).T
varianza = np.einsum("ij,jk,ik->i", jacobiano, cov, jacobiano)
banda = t_critico * np.sqrt(varianza)

izq.fill_between(malla_t, ajuste - banda, ajuste + banda,
                 color=PALETA["rojo"], alpha=0.16, lw=0,
                 label="banda al 95 por ciento del modelo")
izq.plot(malla_t, ajuste, "-", color=PALETA["rojo"], lw=1.4,
         label="modelo de Page calibrado con el lote A")
izq.plot(t_cal, mr_cal, "o", color=PALETA["azul"], ms=5,
         label="lote A, calibración")
izq.plot(t_val, mr_val, "s", color=PALETA["verde"], ms=5,
         label="lote B, validación independiente")
izq.axhline(MR_OBJETIVO, color=PALETA["gris"], ls="--", lw=0.9)
izq.axvspan(t_predicho - u_total, t_predicho + u_total,
            color=PALETA["morado"], alpha=0.16, lw=0)
izq.axvline(t_predicho, color=PALETA["morado"], lw=1.2)
izq.annotate(f"{t_predicho:.2f} h", xy=(t_predicho, MR_OBJETIVO),
             xytext=(t_predicho + 0.55, MR_OBJETIVO + 0.11),
             color=PALETA["morado"], fontsize=9,
             arrowprops={"arrowstyle": "->",
                         "color": PALETA["morado"], "lw": 1.0})
izq.set_xlabel("Tiempo de secado t (h)")
izq.set_ylabel("Razón de humedad MR (adimensional)")
izq.set_xlim(0.0, 8.6)
izq.set_ylim(0.0, 1.05)
izq.set_title("(a) evidencia, dos campañas y el modelo")
izq.legend(loc="upper right", fontsize=7.5)

der.hist(tiempos, bins=70, density=True, color=PALETA["morado"],
         alpha=0.55, edgecolor="none",
         label="Monte Carlo de los parámetros")
for extremo in cobertura:
    der.axvline(extremo, color=PALETA["morado"], ls=":", lw=1.1)
for extremo in (t_predicho - 1.96 * u_lineal,
                t_predicho + 1.96 * u_lineal):
    der.axvline(extremo, color=PALETA["rojo"], ls="--", lw=0.9)
der.axvline(t_predicho, color=PALETA["gris"], lw=1.2)
der.set_xlabel("Tiempo hasta MR igual a 0.10 (h)")
der.set_ylabel("Densidad (1/h)")
der.set_title("(b) margen de la predicción")
der.legend(loc="upper right", fontsize=7.5)

fig.tight_layout()
plt.show()

### 7.1 La afirmación defendible

El capítulo cierra con la obligación que da sentido a todo lo
anterior. Quien entrega el resultado de una simulación a quien decide
debe comunicar el margen, el dominio de validez y aquello que el
modelo no representa, con claridad suficiente para que la decisión se
tome con conocimiento de causa.

In [ ]:
CONCLUSION = {
    "afirmación": (
        f"El secado del maíz de capa delgada a 60 grados Celsius "
        f"alcanza una razón de humedad de {MR_OBJETIVO:.2f} en "
        f"{formato_valor_incertidumbre(t_predicho, u_total, 2, 'h')}, "
        f"con el modelo de Page calibrado sobre el lote A y validado "
        f"sin retoques sobre el lote B."),
    "respaldo": (
        f"En el lote independiente el error cuadrático medio vale "
        f"{m_val['RMSE']:.4f} en razón de humedad y el sesgo "
        f"porcentual {m_val['PBIAS']:.2f} por ciento, dentro del "
        f"criterio declarado antes del ensayo."),
    "dominio de validez": (
        f"Tiempos entre {min(t_cal.min(), t_val.min()):.2f} h y "
        f"{max(t_cal.max(), t_val.max()):.2f} h, a 60 grados Celsius "
        f"y en capa delgada."),
}

LIMITACIONES = [
    ("dominio de validez cubierto por los datos de validación",
     CONCLUSION["dominio de validez"]),
    ("procesos omitidos",
     "encogimiento del grano y gradiente interno de temperatura"),
    ("fuentes de incertidumbre no cuantificadas",
     "variabilidad de la humedad inicial entre lotes de campo"),
    ("dependencia de escenarios futuros no verificables",
     "ninguna, el ensayo es presente y repetible"),
    ("error numérico residual",
     "despreciable, el modelo tiene forma cerrada y el integrador "
     "quedó verificado a orden cuatro"),
]

for titulo, texto in CONCLUSION.items():
    print(f"{titulo.upper()}\n  {texto}\n")
print("LIMITACIONES DECLARADAS")
for titulo, texto in LIMITACIONES:
    print(f"  - {titulo}: {texto}")

assert len(LIMITACIONES) == 5
assert str(round(t_predicho, 2)) in CONCLUSION["afirmación"]

## 8. Ejercicios guiados

Las celdas siguientes llevan la marca `# COMPLETE:` y arrancan con un
valor de partida evidentemente incorrecto, de modo que el cuaderno
corre completo aunque falten por resolver. Al terminarlas, cambie
`REVISAR = True` en la celda de configuración.

### Ejercicio 1. Otra razón de humedad objetivo

El equipo estudia bajar la meta a 0.05. Calcule el tiempo que exige y
su incertidumbre por la ley de primer orden, con las funciones ya
definidas.

In [ ]:
# COMPLETE: use tiempo_objetivo y gradiente_tiempo con mr igual a
# 0.05 y la matriz cov del ajuste, y guarde el tiempo y su
# incertidumbre típica.
t_005 = None
u_005 = None

In [ ]:
# Verificación del ejercicio 1.
verificar("tiempo hasta MR igual a 0.05", t_005, 6.4493, tol=1.0e-4)
verificar("incertidumbre de ese tiempo", u_005, 0.087326, tol=1.0e-3)
if t_005 is not None:
    print(f"\nBajar la meta de 0.10 a 0.05 alarga el secado de "
          f"{t_predicho:.2f} h a {t_005:.2f} h,")
    print(f"y el margen crece de {u_lineal:.3f} h a {u_005:.3f} h, porque")
    print("la extrapolación se aleja de la región mejor determinada.")

### Ejercicio 2. Diseñar dónde medir

La sección 4.6.2 del libro señala que la segunda función del modelo
consiste en decidir dónde medir. Compare la incertidumbre de la
predicción si la próxima campaña añade tres puntos entre 4 h y 6 h
frente a tres puntos entre 0.25 h y 1 h, sin cambiar nada más.

In [ ]:
def u_prediccion_con(t_extra):
    """Incertidumbre de la predicción al añadir instantes al diseño."""
    t_nuevo = np.concatenate([t_cal, np.asarray(t_extra, float)])
    mr_nuevo = np.concatenate([mr_cal, page(np.asarray(t_extra, float),
                                            *theta)])
    # COMPLETE: ajuste el modelo de Page sobre t_nuevo y mr_nuevo,
    # obtenga la nueva matriz de covarianza y devuelva la
    # incertidumbre de la predicción con gradiente_tiempo.
    return None


u_tardios = u_prediccion_con([4.5, 5.0, 5.5])
u_tempranos = u_prediccion_con([0.30, 0.60, 0.90])

In [ ]:
# Verificación del ejercicio 2.
verificar("incertidumbre con puntos tardíos", u_tardios, 0.041867, tol=1.0e-3)
verificar("incertidumbre con puntos tempranos", u_tempranos, 0.050162,
          tol=1.0e-3)
if u_tardios is not None and u_tempranos is not None:
    print(f"\nTres puntos entre 4 h y 6 h dejan la predicción en "
          f"{u_tardios:.4f} h")
    print(f"y tres puntos entre 0.25 h y 1 h la dejan en "
          f"{u_tempranos:.4f} h.")
    mejor = "tardíos" if u_tardios < u_tempranos else "tempranos"
    print(f"Conviene medir en los instantes {mejor}, porque la "
          f"predicción cae cerca de esa región.")

### Ejercicio 3. Validación cruzada del lote A

Cuando los datos escasean, la validación cruzada estima el error de
predicción fuera de la muestra sin sacrificar datos. Calcule el error
del modelo de Page sobre el lote A con cinco bloques y la semilla del
curso, y compárelo con el error observado en el lote B.

In [ ]:
# COMPLETE: divida los índices del lote A en cinco bloques con
# np.array_split sobre una permutación con semilla SEMILLA, ajuste
# con los demás bloques, prediga el que quedó fuera y devuelva la
# raíz del error cuadrático medio de todos los residuales.
rmse_cruzada = None

In [ ]:
# Verificación del ejercicio 3.
verificar("error de validación cruzada", rmse_cruzada, 0.0075229, tol=1.0e-3)
if rmse_cruzada is not None:
    print(f"\nValidación cruzada sobre el lote A {rmse_cruzada:.4f}")
    print(f"Error observado en el lote B      {m_val['RMSE']:.4f}")
    print("El segundo es mayor porque mide algo distinto, el error al")
    print("predecir un lote nuevo y no un dato nuevo del mismo lote.")

### Ejercicio 4. Un criterio de aceptación más exigente

Suponga que el equipo exige ahora un error cuadrático medio inferior
a 0.015 en razón de humedad. Evalúe si el modelo seguiría aceptado y
escriba el veredicto.

In [ ]:
CRITERIO_EXIGENTE = {"RMSE_maximo": 0.015, "PBIAS_maximo": 10.0,
                     "NSE_minimo": 0.95}

# COMPLETE: evalúe las tres condiciones de CRITERIO_EXIGENTE sobre
# m_val y guarde el veredicto en cumple_exigente.
cumple_exigente = None

In [ ]:
# Verificación del ejercicio 4.
if cumple_exigente is None:
    print("[pendiente] la celda marcada COMPLETE sigue sin resolver")
else:
    print(f"Con el criterio original {CRITERIO['RMSE_maximo']:.3f}: "
          f"aceptado {cumple}")
    print(f"Con el criterio exigente {CRITERIO_EXIGENTE['RMSE_maximo']:.3f}: "
          f"aceptado {cumple_exigente}")
    correcta = (cumple_exigente is False)
    print(f"[{'ok' if correcta else 'revisar'}] el error de "
          f"{m_val['RMSE']:.4f} supera el nuevo umbral")
    print("Cambiar el criterio después de ver los datos es exactamente")
    print("lo que el Algoritmo 4.3 prohíbe, y por eso se declara antes.")
    if REVISAR:
        assert correcta

### Ejercicio 5. El guion de la charla

Prepare el guion de la exposición oral con los cuatro momentos que la
sección 4.6.1 del libro recomienda, con la afirmación en el segundo
lugar y no al final.

In [ ]:
# COMPLETE: arme una lista de cuatro parejas con los momentos
# Pregunta, Afirmación, Evidencia y Limitaciones, cada una con su
# contenido tomado de PREGUNTA, CONCLUSION y LIMITACIONES.
guion_charla = None

In [ ]:
# Verificación del ejercicio 5.
if guion_charla is None:
    print("[pendiente] la celda marcada COMPLETE sigue sin resolver")
else:
    for momento, contenido in guion_charla:
        print(f"{momento:14s} {contenido}")
    correcta = (len(guion_charla) == 4
                and guion_charla[1][0] == "Afirmación")
    print(f"\n[{'ok' if correcta else 'revisar'}] la afirmación "
          f"aparece en el segundo momento")
    if REVISAR:
        assert correcta

## 9. Problemas del capítulo

### Problema 4-29, resuelto

Diseñe el protocolo de verificación, calibración y validación de un
modelo de su área, con los datos de cada etapa y el criterio de
aceptación declarado de antemano. La celda siguiente recoge el
protocolo que este cuaderno ejecutó, en la forma en que se firmaría
al comienzo del trabajo.

In [ ]:
PROTOCOLO = pd.DataFrame([
    {"etapa": "Verificación",
     "datos": "ninguno, solución cerrada del modelo de Page",
     "criterio": "orden observado del integrador entre 3.9 y 4.1"},
    {"etapa": "Calibración",
     "datos": "lote A, trece instantes entre 0.25 h y 8 h",
     "criterio": "número de condición por debajo de 5"},
    {"etapa": "Validación",
     "datos": "lote B, diez instantes, campaña independiente",
     "criterio": "RMSE menor que 0.025 y sesgo menor que 10 por ciento"},
    {"etapa": "Incertidumbre",
     "datos": "matriz de covarianza del ajuste y sesgo entre lotes",
     "criterio": "discrepancia entre linealización y Monte Carlo "
                 "menor que 5 por ciento"},
])
with pd.option_context("display.max_colwidth", 56):
    print(PROTOCOLO.to_string(index=False))

def sensibilidad_escalada(modelo, parametros, x, rel=1.0e-6):
    """Columna j igual a theta_j por la derivada, Listado 4.6."""
    J = np.zeros((np.size(x), np.size(parametros)))
    for j in range(np.size(parametros)):
        d = rel * max(abs(parametros[j]), 1.0e-12)
        mas = np.array(parametros, float)
        menos = np.array(parametros, float)
        mas[j], menos[j] = mas[j] + d, menos[j] - d
        J[:, j] = (modelo(x, *mas) - modelo(x, *menos)) / (2 * d) * parametros[j]
    return J


kappa = np.linalg.cond(sensibilidad_escalada(page, theta, t_cal))
cumplimiento = {
    "orden observado del integrador": 3.9 <= p_reg <= 4.1,
    "número de condición de la calibración": kappa < 5.0,
    "criterio de validación": cumple,
    "linealización adecuada": abs(discrepancia) < 5.0,
}
print("\nCumplimiento del protocolo:")
for etapa, cumplida in cumplimiento.items():
    print(f"  {'sí' if cumplida else 'no':3s}  {etapa}")
print(f"\nNúmero de condición {kappa:.2f}, el libro publica 3.18 para "
      f"este mismo ajuste.")
assert all(cumplimiento.values())
assert abs(kappa - 3.18) < 5.0e-3

### Problema 4-2, andamiaje

Un colega afirma que su modelo está validado porque reproduce con
error inferior al 3 por ciento los datos con los que estimó sus
parámetros. Redacte la objeción técnica y proponga el ejercicio que
sí constituiría una validación.

La celda siguiente cuantifica el argumento, comparando el error sobre
los datos de calibración con el error sobre el lote independiente.

In [ ]:
error_relativo_cal = 100 * np.mean(
    np.abs(page(t_cal, *theta) - mr_cal) / np.maximum(mr_cal, 1.0e-6))
error_relativo_val = 100 * np.mean(
    np.abs(mr_predicho - mr_val) / np.maximum(mr_val, 1.0e-6))

print(f"Error relativo medio sobre la calibración "
      f"{error_relativo_cal:.1f} por ciento")
print(f"Error relativo medio sobre el lote independiente "
      f"{error_relativo_val:.1f} por ciento")
print(f"El segundo es {error_relativo_val / error_relativo_cal:.1f} "
      f"veces mayor,")
print("y esa razón es justamente la evidencia que el colega no tiene.")
print("\nLa objeción es que los parámetros se escogieron para minimizar")
print("esa discrepancia, de modo que su pequeñez no prueba nada. El")
print("ejercicio que sí valida es reservar una campaña completa antes")
print("de calibrar, con el criterio escrito y sin retocar parámetros.")
assert error_relativo_val > error_relativo_cal

## Cierre

### Lo que debe saber hacer al terminar

- Escribir la pregunta de investigación y la decisión que apoya antes de tocar los datos.
- Verificar el integrador contra una solución cerrada y medir su orden observado antes de calibrar.
- Calibrar con una campaña y validar con otra independiente, sin retocar parámetros y con criterio declarado de antemano.
- Propagar la incertidumbre de los parámetros a la predicción de interés por linealización y por Monte Carlo, y compararlas.
- Construir la figura única que sostiene la conclusión y redactar la afirmación con su dominio de validez y sus limitaciones.

### Qué revisar en el libro si algo no salió

- Si el orden observado del integrador no da cuatro, revise el Teorema 4.1 y el Algoritmo 4.1.
- Si duda de la independencia del conjunto de validación, revise la Definición 4.7 y el Algoritmo 4.3.
- Si la propagación por linealización y la de Monte Carlo discrepan, revise el Teorema 4.4 y el Algoritmo 4.4.
- Si no sabe qué escribir en las limitaciones, la sección 4.4.2 enumera las cinco obligatorias.
- Si el argumento de la charla no se sostiene en una sola figura, revise el cierre de la sección 4.6.1.

### Declaración del uso de asistentes de programación

Este cuaderno se preparó con apoyo de un asistente de programación.
Todo resultado numérico que aparece aquí se verifica contra el valor
que el libro publica, contra una solución analítica o contra un caso
límite, según recuerda la sección 4.6 del libro. La responsabilidad
del contenido no se transfiere al asistente.